<a href="https://colab.research.google.com/github/matthewhawksby/colabnotebooks/blob/main/Wav2vec2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#pip install -U flash-attn --no-build-isolation

import os
from google.colab import drive
!pip install transformers torchaudio librosa
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
!pip install librosa
import librosa
import requests
import torch, torchaudio

drive.mount('/content/drive')
save_dir = "/content/drive/MyDrive/Colab Notebooks/Final Project CMPT419 - Affective Computing/wav2vec2_large_emotion"

model_name = "facebook/wav2vec2-large-960h"

model = Wav2Vec2ForCTC.from_pretrained(model_name)
processor = Wav2Vec2Processor.from_pretrained(model_name)

model.save_pretrained(save_dir)
processor.save_pretrained(save_dir)
print(f"Model saved to: {save_dir}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-large-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model saved to: /content/drive/MyDrive/Colab Notebooks/Final Project CMPT419 - Affective Computing/wav2vec2_large_emotion


In [2]:

#Get a sample audio file
!wget -O "/content/drive/MyDrive/Colab Notebooks/Final Project CMPT419 - Affective Computing/SampleAudioFiles/sample1.wav" "https://www.voiptroubleshooter.com/open_speech/american/OSR_us_000_0030_8k.wav"

audio_file_path = "/content/drive/MyDrive/Colab Notebooks/Final Project CMPT419 - Affective Computing/SampleAudioFiles/sample1.wav"
waveform, sample_rate = librosa.load(audio_file_path, sr=16000)
print(audio_file_path)

# Resample the audio to 16kHz for wav2vec2
waveform, sample_rate = librosa.load(audio_file_path, sr=16000)
input_values = processor(waveform, return_tensors="pt", sampling_rate=16000).input_values

# Run inference
with torch.no_grad():
    logits = model(input_values).logits

# Speech 2 Text
predicted_ids = torch.argmax(logits, dim=-1)
transcription = processor.batch_decode(predicted_ids)[0]

print("Transcription:", transcription)
print("Logits shape:", logits.shape)


--2025-03-17 05:02:56--  https://www.voiptroubleshooter.com/open_speech/american/OSR_us_000_0030_8k.wav
Resolving www.voiptroubleshooter.com (www.voiptroubleshooter.com)... 162.241.218.124
Connecting to www.voiptroubleshooter.com (www.voiptroubleshooter.com)|162.241.218.124|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 750886 (733K) [audio/x-wav]
Saving to: ‘/content/drive/MyDrive/Colab Notebooks/Final Project CMPT419 - Affective Computing/SampleAudioFiles/sample1.wav’

/content/drive/MyDr 100%[===================>] 733.29K  4.37MB/s    in 0.2s    

2025-03-17 05:02:57 (4.37 MB/s) - ‘/content/drive/MyDrive/Colab Notebooks/Final Project CMPT419 - Affective Computing/SampleAudioFiles/sample1.wav’ saved [750886/750886]

/content/drive/MyDrive/Colab Notebooks/Final Project CMPT419 - Affective Computing/SampleAudioFiles/sample1.wav
Transcription: PAINT THE SOCKETS IN THE WALL DULL GREEN THE CHILD CRAWLED INTO THE DENSE GRASS BRIBES FAIL WHERE HONEST MEN WORK TRAM

In [6]:
#Let's look at the architecture of the model
#print(model)

In [4]:
#Let's look at all named components now.
for name, module in model.named_children():
    print(name, "->", module)

wav2vec2 -> Wav2Vec2Model(
  (feature_extractor): Wav2Vec2FeatureEncoder(
    (conv_layers): ModuleList(
      (0): Wav2Vec2GroupNormConvLayer(
        (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,), bias=False)
        (activation): GELUActivation()
        (layer_norm): GroupNorm(512, 512, eps=1e-05, affine=True)
      )
      (1-4): 4 x Wav2Vec2NoLayerNormConvLayer(
        (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,), bias=False)
        (activation): GELUActivation()
      )
      (5-6): 2 x Wav2Vec2NoLayerNormConvLayer(
        (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,), bias=False)
        (activation): GELUActivation()
      )
    )
  )
  (feature_projection): Wav2Vec2FeatureProjection(
    (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (projection): Linear(in_features=512, out_features=1024, bias=True)
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (encoder): Wav2Vec2Encoder(
    (pos_conv_embed): Wav2Vec2PositionalConv

In [5]:
#And now let's look at all the layers including sublayers.
for name, param in model.named_parameters():
    print(name, param.shape)

wav2vec2.masked_spec_embed torch.Size([1024])
wav2vec2.feature_extractor.conv_layers.0.conv.weight torch.Size([512, 1, 10])
wav2vec2.feature_extractor.conv_layers.0.layer_norm.weight torch.Size([512])
wav2vec2.feature_extractor.conv_layers.0.layer_norm.bias torch.Size([512])
wav2vec2.feature_extractor.conv_layers.1.conv.weight torch.Size([512, 512, 3])
wav2vec2.feature_extractor.conv_layers.2.conv.weight torch.Size([512, 512, 3])
wav2vec2.feature_extractor.conv_layers.3.conv.weight torch.Size([512, 512, 3])
wav2vec2.feature_extractor.conv_layers.4.conv.weight torch.Size([512, 512, 3])
wav2vec2.feature_extractor.conv_layers.5.conv.weight torch.Size([512, 512, 2])
wav2vec2.feature_extractor.conv_layers.6.conv.weight torch.Size([512, 512, 2])
wav2vec2.feature_projection.layer_norm.weight torch.Size([512])
wav2vec2.feature_projection.layer_norm.bias torch.Size([512])
wav2vec2.feature_projection.projection.weight torch.Size([1024, 512])
wav2vec2.feature_projection.projection.bias torch.Size